# Liver Disease — Preprocessing
Encodes Gender, remaps the target to 1/0, imputes missing
values, scales, and splits.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib


In [2]:
df = pd.read_csv("../../data/raw/liver_disease.csv")
df = df.rename(columns={
    "age": "Age",
    "gender": "Gender",
    "tot_bilirubin": "Total_Bilirubin",
    "direct_bilirubin": "Direct_Bilirubin",
    "tot_proteins": "Total_Protiens",
    "albumin": "Albumin",
    "ag_ratio": "Albumin_and_Globulin_Ratio",
    "sgpt": "Alamine_Aminotransferase",
    "sgot": "Aspartate_Aminotransferase",
    "alkphos": "Alkaline_Phosphotase",
    "is_patient": "Dataset",
})
df.shape


(583, 11)

## 1. Encode Gender (Male/Female -> 1/0)

In [3]:
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})


## 2. Remap the target
Original: 1 = disease, 2 = no disease.
We convert to the usual convention: 1 = disease, 0 = no disease.

In [4]:
df["Dataset"] = df["Dataset"].map({1: 1, 2: 0})
df["Dataset"].value_counts()


Dataset
1    416
0    167
Name: count, dtype: int64

## 3. Impute missing values (median)

In [5]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum().sum(), "missing values remaining")


0 missing values remaining


## 4. Separate features/target, split, and scale

In [6]:
X = df.drop("Dataset", axis=1)
y = df["Dataset"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)


Train: (466, 10) | Test: (117, 10)


## 5. Save cleaned data, scaler, and splits

In [7]:
df.to_csv("../../data/processed/liver_disease_cleaned.csv", index=False)
joblib.dump(scaler, "../../models/liver_disease_scaler.pkl")

X_train_scaled.to_csv("../../data/processed/liver_disease_X_train.csv", index=False)
X_test_scaled.to_csv("../../data/processed/liver_disease_X_test.csv", index=False)
y_train.to_csv("../../data/processed/liver_disease_y_train.csv", index=False)
y_test.to_csv("../../data/processed/liver_disease_y_test.csv", index=False)

print("Saved cleaned data, scaler, and train/test splits.")


Saved cleaned data, scaler, and train/test splits.


## Next step
Open **03_model_training.ipynb**.